# Tiny M1 training-paraphrase retry v1

New training from the same pinned base and fresh LoRA, preserving the previous adapters. Each of the original 84 training questions now has two additional newly authored training-only variants, giving 252 distinct questions and 504 answer-only/thinking training rows. Keep development/final questions, constructed fact-chain targets, optimizer/LoRA settings, 120 attempts and all acceptance gates unchanged. No arithmetic evaluation questions are trained on.

Evaluate both development modes/thinking utility at base, post and reload, plus original training wording in both modes at post/reload (870 predictions if the baseline gate passes). Training-wording scores are diagnostic, not acceptance gates or final accuracy. This is a separate tiny diagnostic, not canonical M1/M2.

Use the existing Kaggle GPU kernel and HF_TOKEN. Publish the new source/config/template bank before running the first cell. Preserve the adapter privately and export evidence even if the pilot fails.


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='paraphrase-m1-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/training/paraphrase_data.py').is_file(), 'Publish the new training-paraphrase source/config/bank before running this notebook.'
assert Path('configs/training/m1_tiny_paraphrase_v1.json').is_file()
assert Path('experiments/runs/m1_thinking_mode_diagnostic_20261105/base_utility_sampled_thinking.jsonl').is_file(), 'Publish the verified baseline reference too.'


Cloning into '/kaggle/working/paraphrase-m1-source-c13jwsq0/repo'...


Repository commit: c6f53fe0475b220c8eab1a29dd93336db3507661


In [2]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


2.11.0+cu130 5.20.2 0.21.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], returncode=0)

In [3]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
artifact_repo = 'tharundevulapalli/persistent-llm-unlearning-checkpoints'
api = HfApi(token=hf_token)
assert api.repo_info(artifact_repo, repo_type='model').private
print('Private artifact repository is accessible. Token is not displayed.')


Private artifact repository is accessible. Token is not displayed.


In [4]:
prep_dir = Path(tempfile.mkdtemp(prefix='paraphrase-m1-prepare-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.training.mixed_m1', '--config', 'configs/training/m1_tiny_paraphrase_v1.json', '--output-dir', str(prep_dir), '--mode', 'prepare'], check=True)
report = json.loads((prep_dir / 'masking_report.json').read_text())
audit = json.loads((prep_dir / 'augmentation_audit.json').read_text())
assert report['rows'] == 504 and report['prefixes_verified'] and report['truncated_examples'] == 0
for mode in ['answer_only', 'thinking']:
    assert report['by_mode'][mode]['rows'] == 252 and report['by_mode'][mode]['max_tokens'] <= 256
assert audit['new_templates'] == 14 and audit['unique_training_questions'] == 252
print('Augmentation audit:', json.dumps(audit, indent=2))
rows = [json.loads(line) for line in (prep_dir / 'selected_train.jsonl').read_text().splitlines()]
for row in [r for r in rows if not r['thinking_enabled'] and r['training_question_variant'] != 'original'][:4]:
    print('New training-only question:', row['question'])
example = next(r for r in rows if r['thinking_enabled'] and r['task_type'] == 'author_to_year' and r['training_question_variant'] != 'original')
print('Constructed target:', example['assistant_target'])
print('Original candidate/development/final wording and arithmetic probes remain unchanged.')


{
  "rows": 504,
  "by_mode": {
    "answer_only": {
      "rows": 252,
      "max_tokens": 106,
      "prompt_tokens": 22503,
      "supervised_response_tokens": 1803
    },
    "thinking": {
      "rows": 252,
      "max_tokens": 138,
      "prompt_tokens": 21495,
      "supervised_response_tokens": 6894
    }
  },
  "prefixes_verified": true,
  "truncated_examples": 0,
  "supervision": "All assistant-response tokens and first EOS; thinking includes the constructed rationale."
}
Augmentation audit: {
  "development_rows": 84,
  "new_templates": 14,
  "note": "Existing template bank is consulted only for exclusion, not model input or final-score tuning.",
  "original_bundle_unchanged": true,
  "rows_by_mode": {
    "answer_only": 252,
    "thinking": 252
  },
  "rows_by_partition": {
    "F": 168,
    "R": 336
  },
  "rows_by_question_variant": {
    "original": 168,
    "paraphrase01": 168,
    "paraphrase02": 168
  },
  "template_checks": [
    {
      "max_similarity_to_existing_te

In [5]:
# Actual LoRA training: fresh base + new adapter, not continued training of the old adapter.
output_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-paraphrase-v1-', dir=remote_root))
print('Training outputs:', output_dir, flush=True)
training_env = os.environ.copy()
training_env['HF_TOKEN'] = hf_token
try:
    result = subprocess.run([sys.executable, '-u', '-m', 'src.training.mixed_m1', '--config', 'configs/training/m1_tiny_paraphrase_v1.json', '--output-dir', str(output_dir), '--mode', 'train'], env=training_env)
finally:
    training_env.pop('HF_TOKEN', None)
assert result.returncode in (0, 2), 'Run failed; preserve partial outputs and inspect the traceback before retrying.'
if result.returncode == 2:
    print('Initial base utility/reference gate failed: zero optimizer updates, no adapter created. Preserve the evidence ZIP.')


Training outputs: /kaggle/working/m1-tiny-paraphrase-v1-0ar28u5m


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 485.72it/s]


Thinking=True: generated 10/10 predictions


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions
Mixed training 10/120; response CE=1.1295
Mixed training 20/120; response CE=0.9186
Mixed training 30/120; response CE=0.5712
Mixed training 40/120; response CE=0.3662
Mixed training 50/120; response CE=0.2805
Mixed training 60/120; response CE=0.2249
Mixed training 70/120; response CE=0.1385
Mixed training 80/120; response CE=0.1580
Mixed training 90/120; response CE=0.1451
Mixed training 100/120; response CE=0.0666
Mixed training 110/120; response CE=0.0630
Mixed training 120/120; response CE=0.0449
Thinking=True: generated 10/10 predictions
Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: gener

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 510.93it/s]


Thinking=True: generated 10/10 predictions
Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions
Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions
{
  "execution_complete": true,
  "training_executed": true,
  "canonical_m1": false,
  "original_pilot_result_unchanged": true,
  "pilot": {
    "passed": false,
    "checks": {
      "dev_answer_only/F/author_book": true,
      "dev_answer_only/F/multi_hop": true,
      "dev_answer_only/R/author_book": true,
      "dev_answer_only/R/multi_hop": true,
      "dev_answer_only/utility_baseline_measurable": true,
      "dev_answer_only/util

In [6]:
summary_path = output_dir / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print('Execution complete:', summary['execution_complete'])
    print('Optimizer updates:', summary.get('optimizer_updates', 0))
    print('Pilot:', json.dumps(summary.get('pilot', summary.get('base_utility_gate')), indent=2))
    if 'evaluation' in summary:
        for stage, groups in summary['evaluation'].items():
            for name, group in groups.items():
                print(stage, name, '| answers:', str(group['answer_matches']) + '/' + str(group['count']), '| format:', group['format_violations'], '| caps:', group['token_limit_hits'], '| boundaries:', group['missing_thinking_boundaries'])
else:
    print('No complete summary. Inspect errors and preserve partial logs/checkpoint if available.')
print('This remains a tiny diagnostic. Preserve failed outcomes too.')


Execution complete: True
Optimizer updates: 120
Pilot: {
  "by_fact_mode": {
    "dev_answer_only": {
      "base_utility_accuracy": 1.0,
      "checks": {
        "F/author_book": true,
        "F/multi_hop": true,
        "R/author_book": true,
        "R/multi_hop": true,
        "adapter_changed": true,
        "format": true,
        "generation_limit": true,
        "optimizer_budget": true,
        "save_reload": true,
        "utility_baseline_measurable": true,
        "utility_preserved": true
      },
      "dev": {
        "F/author_book": {
          "accuracy": 1.0,
          "correct": 8,
          "count": 8
        },
        "F/multi_hop": {
          "accuracy": 0.75,
          "correct": 9,
          "count": 12
        },
        "R/author_book": {
          "accuracy": 1.0,
          "correct": 16,
          "count": 16
        },
        "R/multi_hop": {
          "accuracy": 0.9166666666666666,
          "correct": 22,
          "count": 24
        }
      },
  

In [7]:
# Preserve adapter/optimizer/tokenizer and reports privately before the evidence export.
import hashlib
from src.m0_smoke_test import write_json

manifest_path = output_dir / 'artifact_manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    remote_path = 'diagnostics/m1-tiny-paraphrase-v1/' + output_dir.name
    commit = api.upload_folder(repo_id=artifact_repo, repo_type='model', folder_path=str(output_dir), path_in_repo=remote_path, commit_message='Preserve train-paraphrase tiny M1 diagnostic')
    restore_dir = Path(tempfile.mkdtemp(prefix='paraphrase-m1-restore-', dir=remote_root))
    for filename, expected in manifest['artifact_files_sha256'].items():
        restored = hf_hub_download(repo_id=artifact_repo, filename=remote_path + '/' + filename, revision=commit.oid, token=hf_token, local_dir=restore_dir)
        assert hashlib.sha256(Path(restored).read_bytes()).hexdigest() == expected, filename
    receipt = {'repo_id': artifact_repo, 'artifact_revision': commit.oid, 'path': remote_path, 'adapter_file_hashes_verified': True, 'canonical_m1': False}
    write_json(output_dir / 'persistence.json', receipt)
    api.upload_file(repo_id=artifact_repo, repo_type='model', path_or_fileobj=str(output_dir / 'persistence.json'), path_in_repo=remote_path + '/persistence.json', commit_message='Record verified paraphrase adapter round-trip')
    print(json.dumps(receipt, indent=2))
else:
    print('No completed training adapter is available to upload. Preserve partial reports in the evidence ZIP.')


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

diagnostics/m1-tiny-paraphrase-v1/m1-tin(…): reconstructing file:   0%|          |  0.00B / 20.2MB            

diagnostics/m1-tiny-paraphrase-v1/m1-tin(…): downloading bytes:           |  0.00B            

{
  "repo_id": "tharundevulapalli/persistent-llm-unlearning-checkpoints",
  "artifact_revision": "f8fbea97a97cc50f3572c3efb62219dbf5820fa9",
  "path": "diagnostics/m1-tiny-paraphrase-v1/m1-tiny-paraphrase-v1-0ar28u5m",
  "adapter_file_hashes_verified": true,
  "canonical_m1": false
}


In [8]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
print('Adapter/optimizer binaries stay in private Hugging Face storage. Preserve the receipt and ZIP before stopping the runtime.')


Download from Kaggle Output: /kaggle/working/m1-tiny-paraphrase-v1-0ar28u5m-evidence.zip
Adapter/optimizer binaries stay in private Hugging Face storage. Preserve the receipt and ZIP before stopping the runtime.
